# 🦟 Mosquito Breeding Ground Detection
## Comparing YOLOv8 · YOLOv5 · Detectron2 · DETR
**Paper:** *Mosquito Breeding Grounds Detection Using Deep Learning Techniques*  
**Authors:** Varalakshmi Perumal, R.Sasana, Rakshitha P, F.Joevita Faustina Doss  
**Dataset Classes:** `tires`, `water_tanks`, `bottles`, `buckets`, `pools`, `water_tubes`

## ⚙️ Step 0: Setup – Mount Drive & Clone Repo

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone project repository
import os
REPO_DIR = '/content/Mosquito-Breeding-Grounds-Detection'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Joevita20/Mosquito-Breeding-Grounds-Detection-.git {REPO_DIR}
%cd {REPO_DIR}

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q ultralytics pypdf opencv-python-headless seaborn pycocotools timm scipy
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
print('✅ All dependencies installed')

## 📥 Step 2: Download MBG Dataset
> Download from: https://www02.smt.ufrj.br/~tvdigital/database/mosquito/page_01.html
> Place videos in `/content/Mosquito-Breeding-Grounds-Detection/data/raw/videos/`

In [ ]:
# If using Roboflow export (YOLO format), paste your API snippet here:
# from roboflow import Roboflow
# rf = Roboflow(api_key='YOUR_KEY')
# project = rf.workspace().project('mbg-dataset')
# dataset = project.version(1).download('yolov8')

# OR: Extract frames from downloaded videos
!python src/preprocessing/extract_frames.py \
    --video_dir data/raw/videos \
    --output_dir data/frames \
    --fps 1

## 🔄 Step 3: Augment Dataset

In [ ]:
!python src/preprocessing/augment_data.py \
    --input_dir data/frames \
    --output_dir data/augmented \
    --n_augments 5
print('✅ Augmentation complete')

## 🤖 Step 4: Train GAN for Minority Class (Pools)

In [ ]:
!python src/preprocessing/gan_gen.py \
    --real_dir data/augmented/pools \
    --output_dir data/gan_generated/pools \
    --num_images 200 \
    --epochs 200
print('✅ GAN generation complete')

## 📂 Step 5: Convert to YOLO Format

In [ ]:
# If you have COCO-format annotations, convert them:
# !python src/preprocessing/convert_to_yolo_format.py \
#     --annotations data/raw/annotations.json \
#     --images_dir data/augmented \
#     --output_dir data/yolo_format

# Verify data.yaml
!cat data/data.yaml

## 🚀 Step 6a: Train YOLOv8 (SGD, ~110 min)

In [ ]:
from ultralytics import YOLO

model_v8 = YOLO('yolov8n.pt')
results_v8 = model_v8.train(
    data='data/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer='SGD',
    lr0=0.01,
    momentum=0.937,
    device=0,
    project='runs/yolov8',
    name='mbg_train',
    plots=True,
)
print(f'YOLOv8 training done. Saved to: {results_v8.save_dir}')

In [ ]:
# Validate YOLOv8
metrics_v8 = model_v8.val(data='data/data.yaml')
v8_results = {
    'precision': metrics_v8.box.mp,
    'recall': metrics_v8.box.mr,
    'f1': 2 * metrics_v8.box.mp * metrics_v8.box.mr / (metrics_v8.box.mp + metrics_v8.box.mr + 1e-8),
    'map': metrics_v8.box.map50,
}
print(f'YOLOv8 | Precision: {v8_results["precision"]:.3f} | Recall: {v8_results["recall"]:.3f} | mAP: {v8_results["map"]:.3f}')

## 🚀 Step 6b: Train YOLOv5 (SGD, ~91 min)

In [ ]:
import subprocess, sys
if not os.path.exists('yolov5'):
    !git clone https://github.com/ultralytics/yolov5
    !pip install -q -r yolov5/requirements.txt

!python yolov5/train.py \
    --data data/data.yaml \
    --weights yolov5s.pt \
    --epochs 100 \
    --batch-size 16 \
    --imgsz 640 \
    --optimizer SGD \
    --device 0 \
    --project runs/yolov5 \
    --name mbg_train

In [ ]:
# Validate YOLOv5
!python yolov5/val.py \
    --weights runs/yolov5/mbg_train/weights/best.pt \
    --data data/data.yaml \
    --device 0

## 🚀 Step 6c: Train Detectron2 (Adam/Adagrad, ~60 min)

In [ ]:
!python models/detectron2/train_detectron2.py \
    --data_dir data/detectron2_format \
    --optimizer adam \
    --epochs 60 \
    --device cuda

## 🚀 Step 6d: Train DETR (~140 min)

In [ ]:
!python models/detr/train_detr.py \
    --data_dir data/detr_format \
    --epochs 140 \
    --device cuda

## 📊 Step 7: Compare All Models (Reproduce Table I from Paper)

In [ ]:
import sys
sys.path.insert(0, '.')
from src.evaluation.plot_results import plot_map_comparison, PAPER_RESULTS
from src.evaluation.metrics import print_metrics_table

# Fill these in with your actual validation results:
YOUR_RESULTS = {
    'YOLOv8':   v8_results,  # filled above
    # 'YOLOv5': {'precision': ..., 'recall': ..., 'f1': ..., 'map': ...},
    # 'Detectron2': {...},
    # 'DETR': {...},
}

print('\n📄 PAPER RESULTS (Table I):')
print_metrics_table(PAPER_RESULTS)

print('\n🧪 YOUR RESULTS:')
print_metrics_table(YOUR_RESULTS)

plot_map_comparison(YOUR_RESULTS, output_path='results/map_comparison.png')
from IPython.display import Image
Image('results/map_comparison.png')

## 🔍 Step 8: Run Inference on New Image

In [ ]:
from ultralytics import YOLO
from IPython.display import Image

model = YOLO('runs/yolov8/mbg_train/weights/best.pt')

# Change to any image path
TEST_IMAGE = 'data/yolo_format/test/images/your_image.jpg'
results = model.predict(TEST_IMAGE, conf=0.25, save=True, project='runs/inference')

# Show result
import glob
latest = sorted(glob.glob('runs/inference/**/*.jpg', recursive=True))[-1]
Image(latest)

## ✅ Step 9: Save Results to GitHub

In [ ]:
!git add results/ runs/
!git commit -m 'Add model training results and comparisons'
!git push